# Module 4: Exceptions and Context Managers

**Concepts:** `try/except` · raising errors · custom exceptions · `with`

> **Safety note:** This is a **programming** exercise. It is **not** medical advice, not a diagnosis, and **not** for real patients or real medicines. We use invented drug names (for example **Alercin**) and school biology/chemistry only.

### How to use this notebook

1. Read the markdown cell (the story).
2. Run the code cell under it with **Shift+Enter**.
3. Check **What you should see**.
4. Try the exercises before opening **Solutions**.

**Idea:** Fail clearly. Never hide a bad dose or a bad DNA letter.

## 1. Why exceptions?

Returning `None` on every failure forces callers to remember checks. Exceptions bubble up until handled.

**What you should see:** a friendly message instead of a crash for bad text.

In [ ]:
text = "not-a-number"
try:
    value = float(text)
except ValueError:
    print("could not read temperature (practice)")
else:
    print("temp", value)

## 2. `try` / `except` / `else` / `finally`

- **except** — handle expected failures  
- **else** — runs if no exception  
- **finally** — always runs (cleanup)

**What you should see:** `finally` always prints.

In [ ]:
def read_temp(text):
    try:
        t = float(text)
    except ValueError:
        print("bad temperature text")
        return None
    else:
        print("parsed OK")
        return t
    finally:
        print("device check finished")


print("result", read_temp("37.6"))
print("---")
print("result", read_temp("hot"))

## 3. Catch specific errors

Avoid bare `except:`. Catch `ValueError`, `KeyError`, etc.

**What you should see:** missing key handled.

In [ ]:
chart = {"name": "Ana"}
try:
    print(chart["temp_c"])
except KeyError:
    print("temperature missing from chart")

## 4. Custom errors — InvalidDoseError

**What you should see:** error message for weight ≤ 0 and for OVER_MAX style raise.

In [ ]:
class ClinicError(Exception):
    # Base error for our practice clinic code.
    pass


class InvalidDoseError(ClinicError):
    pass


def dose_mg(weight_kg, mg_per_kg=10, max_mg=1000):
    if weight_kg <= 0:
        raise InvalidDoseError("weight must be positive (practice)")
    needed = weight_kg * mg_per_kg
    if needed > max_mg:
        raise InvalidDoseError("OVER_MAX (practice)")
    return needed


for w in (52.5, -1, 120):
    try:
        print(w, "->", dose_mg(w))
    except InvalidDoseError as e:
        print(w, "->", e)

## 5. Bad DNA base → ValueError

**What you should see:** complement works, then clear error for `X`.

In [ ]:
PAIRS = {"A": "T", "T": "A", "C": "G", "G": "C"}


def complement(dna):
    dna = dna.upper().replace(" ", "")
    out = []
    for b in dna:
        if b not in PAIRS:
            raise ValueError(f"unknown DNA base: {b}")
        out.append(PAIRS[b])
    return "".join(out)


print(complement("ATGC"))
try:
    complement("ATGX")
except ValueError as e:
    print(e)

## 6. `with` — lab notebook always closes

**What you should see:** file contents, and file is closed even after a fake error path.

In [ ]:
from pathlib import Path

path = Path("_lab_note_m4.txt")
with path.open("w", encoding="utf-8") as f:
    f.write("shift note: practice only\n")

with path.open("r", encoding="utf-8") as f:
    print(f.read().strip())

print("closed?", f.closed)

## 7. Class context manager — monitor session

`__enter__` / `__exit__` always disconnect.

**What you should see:** connected → reading → disconnected.

In [ ]:
class MonitorSession:
    def __enter__(self):
        print("monitor connected (practice)")
        return self

    def read_pulse(self):
        return 88

    def __exit__(self, exc_type, exc, tb):
        print("monitor disconnected (practice)")
        return False  # do not swallow errors


with MonitorSession() as mon:
    print("pulse", mon.read_pulse())

## 8. Capstone — dose + DNA in one safe block

**What you should see:** success path and two handled errors.

In [ ]:
class InvalidDoseError(Exception):
    pass


PAIRS = {"A": "T", "T": "A", "C": "G", "G": "C"}


def dose_mg(weight_kg, mg_per_kg=10, max_mg=1000):
    if weight_kg <= 0:
        raise InvalidDoseError("bad weight")
    needed = weight_kg * mg_per_kg
    if needed > max_mg:
        raise InvalidDoseError("OVER_MAX")
    return needed


def complement_safe(dna):
    out = []
    for b in dna.upper():
        if b not in PAIRS:
            raise ValueError(f"bad base {b}")
        out.append(PAIRS[b])
    return "".join(out)


for label, action in [
    ("dose", lambda: dose_mg(52.5)),
    ("dose bad", lambda: dose_mg(0)),
    ("dna", lambda: complement_safe("ATGC")),
    ("dna bad", lambda: complement_safe("ATGX")),
]:
    try:
        print(label, "->", action())
    except (InvalidDoseError, ValueError) as e:
        print(label, "->", type(e).__name__, e)

---
# Exercises

**Ex 1** — `require_key(chart, key)` raises `KeyError` with a clear message if missing.

In [ ]:
# TODO

**Ex 2** — `try/except` around `int("not-a-port")` — print a friendly message (we pretend a ward number).

In [ ]:
# TODO

**Ex 3** — `@contextmanager` timer printing elapsed seconds (`time.perf_counter`).

In [ ]:
# TODO
# from contextlib import contextmanager
# import time

**Ex 4** — `RollbackError(ClinicError)`; catch as `ClinicError`.

In [ ]:
# TODO

---
# Solutions

In [ ]:
def require_key(chart, key):
    if key not in chart:
        raise KeyError(f"missing {key} in chart")
    return chart[key]


print(require_key({"name": "Ana"}, "name"))
try:
    require_key({"name": "Ana"}, "temp_c")
except KeyError as e:
    print(e)

In [ ]:
try:
    int("not-a-port")
except ValueError:
    print("ward number must be digits (practice)")

In [ ]:
from contextlib import contextmanager
import time


@contextmanager
def timer(name):
    start = time.perf_counter()
    try:
        yield
    finally:
        print(name, "took", round(time.perf_counter() - start, 4), "s")


with timer("fake-work"):
    sum(range(10000))

In [ ]:
class ClinicError(Exception):
    pass


class RollbackError(ClinicError):
    pass


def deploy_practice():
    raise RollbackError("undo practice change")


try:
    deploy_practice()
except ClinicError as e:
    print("caught", type(e).__name__, e)

**Next:** Module 5 — files, regex, logging.